In [1]:
# Importing libraries
import pandas as pd
import numpy as np
import os

In [3]:
# Loading raw data
df21 = pd.read_csv('../data/raw/tn_2021_results.csv')
df26 = pd.read_csv('../data/raw/tn_2026_results.csv')
cm   = pd.read_csv('../data/raw/constituency_master.csv')

print("2021 shape:", df21.shape)
print("2026 shape:", df26.shape)
print("Master shape:", cm.shape)

2021 shape: (4232, 8)
2026 shape: (4257, 8)
Master shape: (234, 5)


In [5]:
# First look of the data
print("=== 2021 SAMPLE ===")
print(df21.head(3))

print("\n=== 2026 SAMPLE ===")
print(df26.head(3))

print("\n=== MASTER SAMPLE ===")
print(cm.head(3))

=== 2021 SAMPLE ===
   constituency  ac_number         candidate party   votes  turnout reserved  \
0  Gummidipundi          1  GOVINDARAJAN T.J   DMK  126452    78.08      GEN   
1  Gummidipundi          1         PRAKASH M   PMK   75514    78.08      GEN   
2  Gummidipundi          1              USHA   NTK   11701    78.08      GEN   

          region  
0  Chennai Metro  
1  Chennai Metro  
2  Chennai Metro  

=== 2026 SAMPLE ===
    constituency  ac_number         candidate   party  votes  turnout  \
0  Gummidipoondi          1     S.VIJAYAKUMAR     TVK  94320      NaN   
1  Gummidipoondi          1        SUDHAKAR.V  AIADMK  66375      NaN   
2  Gummidipoondi          1  T.J.GOVINDARAJAN     DMK  62492      NaN   

  reserved         region  
0      GEN  Chennai Metro  
1      GEN  Chennai Metro  
2      GEN  Chennai Metro  

=== MASTER SAMPLE ===
   ac_number  constituency    district         region reserved
0          1  Gummidipundi  Tiruvallur  Chennai Metro      GEN
1       

In [7]:
# Data quality audit
print("=== NULL VALUES ===")
print("\n2021 nulls:")
print(df21.isnull().sum())

print("\n2026 nulls:")
print(df26.isnull().sum())

print("\n=== DATA TYPES ===")
print("\n2021 dtypes:")
print(df21.dtypes)

print("\n2026 dtypes:")
print(df26.dtypes)

print("\n=== DUPLICATE ROWS ===")
print(f"2021 duplicates: {df21.duplicated().sum()}")
print(f"2026 duplicates: {df26.duplicated().sum()}")

=== NULL VALUES ===

2021 nulls:
constituency    0
ac_number       0
candidate       0
party           0
votes           0
turnout         0
reserved        0
region          0
dtype: int64

2026 nulls:
constituency       0
ac_number          0
candidate          0
party              0
votes              0
turnout         4257
reserved           0
region             0
dtype: int64

=== DATA TYPES ===

2021 dtypes:
constituency     object
ac_number         int64
candidate        object
party            object
votes             int64
turnout         float64
reserved         object
region           object
dtype: object

2026 dtypes:
constituency     object
ac_number         int64
candidate        object
party            object
votes             int64
turnout         float64
reserved         object
region           object
dtype: object

=== DUPLICATE ROWS ===
2021 duplicates: 0
2026 duplicates: 0


In [9]:
# Name mismatch check (this is why we will use ac_number(primary key))
names_21 = set(df21['constituency'].str.strip().unique())
names_26 = set(df26['constituency'].str.strip().unique())

in_21_not_26 = names_21 - names_26
in_26_not_21 = names_26 - names_21

print(f"Names in 2021 but NOT in 2026: {len(in_21_not_26)}")
print(f"Names in 2026 but NOT in 2021: {len(in_26_not_21)}")
print("\nExample mismatches:")
for a, b in zip(list(in_21_not_26)[:5], list(in_26_not_21)[:5]):
    print(f"  2021: {a}  |  2026: {b}")

Names in 2021 but NOT in 2026: 34
Names in 2026 but NOT in 2021: 34

Example mismatches:
  2021: Nilakottai  |  2026: Thiruthuraipoondi
  2021: Gummidipundi  |  2026: Edappadi
  2021: Coimbatore South  |  2026: Coimbatore (North)
  2021: Arantangi  |  2026: Gummidipoondi
  2021: Gudiyatham  |  2026: Sholingur


In [11]:
# Standardizing party names for cleaner analysis
party_map = {
    'Bahujan Samaj Party' : 'BSP',
    'Rashtriya Janata Dal': 'RJD',
    'Communist Party of India (Marxist-Leninist) (Liberation)': 'CPI(ML)',
    'Communist Party of India (Marxist-Leninist) Red Star'    : 'CPI(ML)',
    'All India Forward Bloc'                                  : 'AIFB',
    'Indian National League'                                  : 'INL',
}

df26['party'] = df26['party'].replace(party_map)
df21['party'] = df21['party'].replace(party_map)

print("Unique parties in 2026:", df26['party'].nunique())
print("Unique parties in 2021:", df21['party'].nunique())

Unique parties in 2026: 104
Unique parties in 2021: 109


In [13]:
# Extracting constituency-level winners

def get_winners(df):
    """
    Input : candidate-level dataframe (many rows per constituency)
    Output: one row per constituency showing the winner
    """
    # Step A: finding the index of the highest-vote candidate per constituency
    winner_idx = df.groupby('ac_number')['votes'].idxmax()
    winners = df.loc[winner_idx].copy()

    # Step B: calculating total valid votes per constituency
    total_votes = df.groupby('ac_number')['votes'].sum().rename('total_votes')
    winners = winners.join(total_votes, on='ac_number')

    # Step C: calculating winner's vote share percentage
    winners['vote_share'] = (winners['votes'] / winners['total_votes'] * 100).round(2)

    return winners.reset_index(drop=True)


w21 = get_winners(df21)
w26 = get_winners(df26)

print("2021 winners shape:", w21.shape)   
print("2026 winners shape:", w26.shape)   
print("\nSample winner row:")
print(w26[['ac_number','constituency','party','votes','total_votes','vote_share']].head(3))

2021 winners shape: (234, 10)
2026 winners shape: (234, 10)

Sample winner row:
   ac_number   constituency   party   votes  total_votes  vote_share
0          1  Gummidipoondi     TVK   94320       232522       40.56
1          2        Ponneri     TVK  110439       226819       48.69
2          3      Tiruttani  AIADMK   89169       238798       37.34


In [15]:
# Calculating winning margins (winner votes minus runner-up votes)

def get_margins(df):
    """
    Returns margin of victory for each constituency
    """
    # Sorting by constituency and votes descending
    sorted_df = df.sort_values(['ac_number', 'votes'], ascending=[True, False])

    # Taking top 2 candidates per constituency
    top2 = sorted_df.groupby('ac_number').head(2)

    # Subtracting 2nd place from 1st place
    margins = top2.groupby('ac_number')['votes'].agg(list).reset_index()
    margins['margin']       = margins['votes'].apply(lambda x: x[0] - x[1] if len(x) > 1 else x[0])
    margins['winner_votes'] = margins['votes'].apply(lambda x: x[0])

    # Adding total votes for percentage calculation
    totals = df.groupby('ac_number')['votes'].sum().rename('total_votes')
    margins = margins.join(totals, on='ac_number')
    margins['margin_pct'] = (margins['margin'] / margins['total_votes'] * 100).round(2)

    return margins[['ac_number', 'margin', 'winner_votes', 'total_votes', 'margin_pct']]


m21 = get_margins(df21)
m26 = get_margins(df26)

print("Avg margin 2021:", round(m21['margin'].mean()))
print("Avg margin 2026:", round(m26['margin'].mean()))
print("\nNarrowest win in 2026:")
print(m26.nsmallest(3, 'margin')[['ac_number','margin','margin_pct']])

Avg margin 2021: 22871
Avg margin 2026: 16784

Narrowest win in 2026:
     ac_number  margin  margin_pct
184        185       1        0.00
53          54     138        0.06
228        229     214        0.09


In [17]:
# Building master table joining everything together
# This single table will power ALL our charts and stories

master = (
    w21[['ac_number', 'party', 'vote_share', 'turnout', 'reserved', 'region']]
    .rename(columns={
        'party'      : 'party_21',
        'vote_share' : 'share_21',
        'turnout'    : 'turnout_21'
    })
    .merge(
        w26[['ac_number', 'party', 'vote_share']]
        .rename(columns={
            'party'      : 'party_26',
            'vote_share' : 'share_26'
        }),
        on='ac_number'
    )
    .merge(
        cm[['ac_number', 'constituency', 'district', 'region']]
        .rename(columns={'region': 'region_master'}),
        on='ac_number'
    )
    .merge(m21[['ac_number', 'margin', 'margin_pct']]
        .rename(columns={'margin': 'margin_21', 'margin_pct': 'mpct_21'}),
        on='ac_number'
    )
    .merge(m26[['ac_number', 'margin', 'margin_pct']]
        .rename(columns={'margin': 'margin_26', 'margin_pct': 'mpct_26'}),
        on='ac_number'
    )
)

# Adding derived columns
master['flipped']       = master['party_21'] != master['party_26']
master['margin_change'] = master['margin_26'] - master['margin_21']

print("Master table shape:", master.shape)   
print("\nColumns:", master.columns.tolist())
print("\nSample:")
print(master.head(3))

Master table shape: (234, 17)

Columns: ['ac_number', 'party_21', 'share_21', 'turnout_21', 'reserved', 'region', 'party_26', 'share_26', 'constituency', 'district', 'region_master', 'margin_21', 'mpct_21', 'margin_26', 'mpct_26', 'flipped', 'margin_change']

Sample:
   ac_number party_21  share_21  turnout_21 reserved         region party_26  \
0          1      DMK     56.94       78.08      GEN  Chennai Metro      TVK   
1          2      INC     44.94       78.20       SC  Chennai Metro      TVK   
2          3      DMK     51.72       78.76      GEN  Chennai Metro   AIADMK   

   share_26  constituency    district  region_master  margin_21  mpct_21  \
0     40.56  Gummidipundi  Tiruvallur  Chennai Metro      50938    22.94   
1     48.69       Ponneri  Tiruvallur  Chennai Metro       9689     4.61   
2     37.34     Tiruttani  Tiruvallur  Chennai Metro      29253    12.58   

   margin_26  mpct_26  flipped  margin_change  
0      27945    12.02     True         -22993  
1      557

In [19]:
# Exporting cleaned files for reuse in Power BI and future notebooks
os.makedirs('../data/cleaned', exist_ok=True)

master.to_csv('../data/cleaned/master_analysis.csv', index=False)
w26.to_csv('../data/cleaned/winners_2026.csv', index=False)
w21.to_csv('../data/cleaned/winners_2021.csv', index=False)

print("All cleaned files saved to data/cleaned/")
print(f"Master table: {master.shape[0]} rows x {master.shape[1]} columns")

All cleaned files saved to data/cleaned/
Master table: 234 rows x 17 columns
